# Chapter 5 &mdash; Running and Testing DFA with `nthnumeric`

**Concept 14 of the Chapter 5 decomposition:** *Running and Testing DFA in Jove with `nthnumeric`*

Generate the first $n$ strings in numeric order and check `accepts_dfa` on each &mdash; short strings first, where the bugs are.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Testing-With-Nthnumeric/Concept-Testing-With-Nthnumeric.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.LangDef        import *
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


`nthnumeric(i, S)` returns the $i$-th string over alphabet `S` in **numeric order**:
$\varepsilon$, then all length-1, then all length-2, and so on. It takes a **list**,
not a set &mdash; a set has no order, and the function asserts on one.

Numeric order is the right test order because **short strings come first**, and that is
where specification bugs live ($\varepsilon$, single symbols, the first pair).

The standard harness: enumerate, compare `accepts_dfa` against a reference predicate,
and print the **first** disagreement rather than a count.

## 2. Definitions

### The enumeration

In [ ]:
def numeric(n, sigma=['0','1']):
    return [nthnumeric(i, sigma) for i in range(n)]

print(numeric(12))

### The harness: first disagreement wins

In [ ]:
def check(D, spec, n=512, sigma=['0','1']):
    for i in range(n):
        s = nthnumeric(i, sigma)
        if accepts_dfa(D, s) != spec(s):
            return i, s, accepts_dfa(D, s), spec(s)
    return None

## 3. Tests

A correct machine passes silently.

In [ ]:
D = md2mc('''DFA
IF : 0 -> Odd
IF : 1 -> IF
Odd: 0 -> IF
Odd: 1 -> Odd
''')
r = check(D, lambda s: s.count('0') % 2 == 0)
print("first disagreement :", r)
assert r is None
print("checked the first 512 strings in numeric order -- all agree")

A buggy machine is caught, and the witness is **short** because of the ordering.

In [ ]:
Bug = md2mc('''DFA
IF : 0 -> Odd
IF : 1 -> Odd      !! BUG: 1s should not flip parity
Odd: 0 -> IF
Odd: 1 -> IF
''')
i, s, got, want = check(Bug, lambda s: s.count('0') % 2 == 0)
print("first disagreement at index %d: s=%r  dfa=%s  spec=%s" % (i, s, got, want))
assert i < 8, "numeric order should find it almost immediately"

`nthnumeric` needs a **list**; a set raises.

In [ ]:
try:
    nthnumeric(5, {'0','1'})
except AssertionError as e:
    print("AssertionError:", str(e)[:80])
print("\nA set has no order, so 'the i-th string' would be meaningless.")

The alphabet is yours to choose, but it must have **exactly two** symbols.

In [ ]:
print("over ['a','b'] :", [nthnumeric(i, ['a','b']) for i in range(12)])
try:
    nthnumeric(5, ['a','b','c'])
except AssertionError as e:
    print("\nthree symbols ->", str(e)[:60])
print("nthnumeric enumerates in BINARY numeric order; for a larger alphabet,")
print("use itertools.product grouped by length.")

## 4. Animation

The machine under test.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(D, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Instrument `check` to report **all** disagreements up to length 6. How many?
2. Why is numeric order better than testing random strings of length 20?
3. Use `check` on a DFA you wrote in Chapter 4. Did it pass?

In [ ]:
# Your work for the exercises above.